<a href="https://colab.research.google.com/github/HaomingJu/Colab/blob/main/boyou_320x160.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step.1 安装环境

In [1]:
%pip install ultralytics
from ultralytics import YOLO, checks, hub
import torch

checks()  # Verify system setup for Ultralytics training

if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Available GPUs: {gpu_count}")
    for i in range(gpu_count):
        print(f"GPU ID: {i}, Name: {torch.cuda.get_device_name(i)}")
else:
    print("No GPU available.")


Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.9/112.6 GB disk)
Available GPUs: 1
GPU ID: 0, Name: Tesla T4


# Step.2 挂载Gooogle Driver

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Step.3 登录Ultralytics

In [3]:

from google.colab import userdata
api_key = userdata.get('UltralyticsTOKEN')
# Login to HUB using your API key (https://hub.ultralytics.com/settings?tab=api+keys)
ret_login = hub.login(api_key)
if not ret_login:
  print("Login Ultralytics failed.")

Ultralytics HUB: New authentication successful ✅


# Step.4 博优

## Step.4.1 数据准备

In [5]:
!unzip /content/drive/MyDrive/data/博优数据/boyou_part2.zip

Archive:  /content/drive/MyDrive/data/博优数据/boyou_part2.zip
replace boyou_part2/detect/images/val/43.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

## Step.4.2 训练

In [6]:
from ultralytics.utils import SETTINGS
SETTINGS['hub'] = False # Temporarily disable Ultralytics HUB to fix PosixPath error

model = YOLO('yolo11l')

results = model.train(
    data='/content/boyou_part2/detect/detect.yaml',  # 数据集配置
    imgsz=320,                 # (高度x宽度) 注意顺序
    epochs=5,
    device=0,
    project='/content/drive/MyDrive/boyou_320x160/runs/train',
    name='boyou_320x160',
    exist_ok=True,                  # 关键参数：启用HUB同步
)

Ultralytics 8.4.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/boyou_part2/detect/detect.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=boyou_320x160, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, 

## Step.4.3 量化模型导出量化



In [ ]:
model_pt_jushanghui = YOLO("/content/drive/MyDrive/boyou_320x160/runs/train/boyou_320x160/weights/best.pt")
model_pt_jushanghui.export(
    format="saved_model",
    data='/content/boyou_part2/detect/detect.yaml',  # Dataset configuration used for training
    imgsz=320,
    int8=True,
    nms=True,
    device=0)



## Step.4.4 验证

In [ ]:
tf_savedmodel_model = YOLO("/content/drive/MyDrive/boyou_320x160/runs/train/boyou_320x160/weights/best_saved_model")

results = tf_savedmodel_model("https://ultralytics.com/images/bus.jpg")

# Step.N 模型部署

## Step.N.1 容器启动

In [ ]:
#!/usr/bin/env bash

docker run -it -d \
        -p 8030:8500 \
        -p 8011:8501 \
        --restart=always \
        --gpus=all   \
        --cpus=32 \
        --memory=32g \
        --name=model_jushanghui_int8 \
        -v /home/haoming/models/jushanghui_yolo11l_360x360_v1_saved_model:/models/model_jushanghui_int8 \
        -e MODEL_NAME=model_jushanghui_int8    \
        -t tensorflow/serving:latest-gpu \
        --enable_batching=true


docker run -it -d \
        -p 8034:8500 \
        -p 8015:8501 \
        --restart=always \
        --gpus=all   \
        --cpus=32 \
        --memory=32g \
        --name=model_boyou_int8 \
        -v /home/haoming/models/model_boyou_int8:/models/model_boyou_int8 \
        -e MODEL_NAME=model_boyou_int8    \
        -t tensorflow/serving:latest-gpu \
        --enable_batching=true